# Stage 4: 2D multi-channel transport and conductance quantization

This walkthrough uses the Stage 4 public functions and saved clean-strip sweep. Site ordering is `index(x, y) = x*Ly + y`; both device directions have open boundaries, while the leads are semi-infinite along x and hard-wall across y. The Hamiltonian has onsite `epsilon_0` and nearest-neighbour hopping `-t`.

In [ ]:
from pathlib import Path
import csv
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "results/stage4_multichannel.csv").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from quantum_transport.two_dimensional import rectangular_device_hamiltonian, transverse_slice_hamiltonian
from quantum_transport.modes import transverse_modes, open_mode_count
from quantum_transport.two_dimensional_leads import two_dimensional_surface_green_function, two_dimensional_self_energy_matrices
from quantum_transport.two_dimensional_transport import (
    broadening_matrix, retarded_2d_device_green_function, multichannel_transmission,
    physical_conductance,
)

Lx, Ly, t, epsilon_0, t_c = 12, 4, 1.0, 0.0, 1.0
H_device = rectangular_device_hamiltonian(Lx, Ly, epsilon_0, t)
print("device shape:", H_device.shape)
print("site index convention: index(x,y) = x*Ly + y")

## 1. The 2D lattice and device Hamiltonian

The site-ordering convention makes each transverse slice contiguous in the dense matrix. The drawing shows the rectangular geometry; the matrix adds only x- and y-neighbour hopping.

In [ ]:
x_sites, y_sites = np.meshgrid(np.arange(Lx), np.arange(Ly), indexing="ij")
fig, ax = plt.subplots(figsize=(8, 2.8))
for x in range(Lx):
    for y in range(Ly):
        if x + 1 < Lx:
            ax.plot([x, x + 1], [y, y], color="0.75", linewidth=0.6)
        if y + 1 < Ly:
            ax.plot([x, x], [y, y + 1], color="0.75", linewidth=0.6)
ax.scatter(x_sites.ravel(), y_sites.ravel(), s=14, color="tab:blue")
ax.set(xlabel="Longitudinal slice x", ylabel="Transverse site y", title=f"Open-boundary strip: {Lx} x {Ly}")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 2?3. Transverse slice and mode spectrum

The transverse Hamiltonian is an open chain. Its numerical eigenvalues agree with `epsilon_0 - 2t cos(n pi/(Ly+1))`; each eigenvector is one transverse mode.

In [ ]:
H_y = transverse_slice_hamiltonian(Ly, epsilon_0, t)
mode_energies, mode_vectors = transverse_modes(Ly, epsilon_0, t)
mode_indices = np.arange(1, Ly + 1)
analytic_mode_energies = epsilon_0 - 2*t*np.cos(mode_indices*np.pi/(Ly+1))
print("H_y =\n", H_y)
print("numerical mode energies:", mode_energies)
print("analytic mode energies:", analytic_mode_energies)
print("mode eigenvector matrix shape:", mode_vectors.shape)

## 4?5. Semi-infinite lead surface Green function and self-energy

For each transverse eigenenergy `epsilon_n`, the semi-infinite 1D surface Green function is evaluated using the Stage 3 retarded branch. The mode-space diagonal matrix is transformed back to transverse site coordinates. The left and right self-energies place `t_c^2 g_surface` only on the first and last slices.

In [ ]:
energy = 0.0
surface = two_dimensional_surface_green_function(energy, Ly, epsilon_0, t)
sigma_left, sigma_right = two_dimensional_self_energy_matrices(
    energy, Lx, Ly, epsilon_0, t, t_c
)
print("surface Green matrix shape:", surface.shape)
print("surface Green matrix at E=0:\n", surface)
print("embedded self-energy shapes:", sigma_left.shape, sigma_right.shape)
print("left nonzero boundary block:\n", sigma_left[:Ly, :Ly])

## 6?8. Device Green function, broadening, and Landauer transmission

The device Green function is obtained by solving the matrix equation against the identity. Broadening is `Gamma = i(Sigma - Sigma^dagger)`. The total transmission sums all open modes through the multi-channel Landauer trace.

In [ ]:
G_r = retarded_2d_device_green_function(energy, H_device, sigma_left, sigma_right)
Gamma_L, Gamma_R = broadening_matrix(sigma_left), broadening_matrix(sigma_right)
T = multichannel_transmission(G_r, Gamma_L, Gamma_R)
print("G^r shape:", G_r.shape)
print("Gamma shapes:", Gamma_L.shape, Gamma_R.shape)
print("T(E=0):", T)
print("G(E=0) [S]:", physical_conductance(T))

## 9?10. Open-channel count and clean matched-strip comparison

A mode is propagating for `|E - epsilon_n| <= 2t`. The numerical `T(E)` is not clipped or normalized; its agreement with the analytic `N_open(E)` follows from the matched device and semi-infinite leads. The saved sweep excludes energies within 0.05 tight-binding units of any threshold when reporting its primary error summary.

In [ ]:
N_open_at_energy = open_mode_count(energy, mode_energies, t)
print("N_open(E=0):", N_open_at_energy)
with (ROOT / "results/stage4_multichannel.csv").open(newline="") as stream:
    sweep = list(csv.DictReader(stream))
print("saved sweep rows:", len(sweep))
for row in sweep:
    if abs(float(row["energy"]) - energy) < 1e-12:
        print("saved E=0 result:", row)
        break

In [ ]:
for filename in (
    "stage4_mode_thresholds.png",
    "stage4_multichannel_transport.png",
    "stage4_conductance.png",
):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.imshow(plt.imread(ROOT / "figures" / filename))
    ax.axis("off")
    ax.set_title(filename)
    plt.show()